# RHNA & Housing Production Validation Check 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
PROCESSED = Path("../data/processed").resolve()

print(PROCESSED)
for f in PROCESSED.glob("*.csv"):
    print(f.name)


/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed
rhna_housing_production_2025_regional_total.csv
powerbi_production_by_housing_type_2018_2025.csv
rhna_housing_production_validation_report.csv
powerbi_rhna_production_2018_2025_long.csv
powerbi_dof_annual_housing_stock_2020_2025.csv
powerbi_housing_stock_benchmarks_2000_2010_2021_2024.csv
apr_production_2018_2025_by_jurisdiction_year.csv
rhna_housing_production_2025_by_jurisdiction.csv


In [3]:
# Load RHNA jurisdiction output
rhna_file = PROCESSED / "rhna_housing_production_2025_by_jurisdiction.csv"
rhna = pd.read_csv(rhna_file)

print(rhna.shape)
rhna.head()


(19, 61)


,jur_clean,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,ent_low_total,...,rhna_remaining_total,rhna_pct_achieved,population_total,population_household,housing_units_total,occupied_units,vacant_units,single_family_units,multifamily_units,mobile_home_units
0,carlsbad,202,343,718,13,36,141,428,11,0,...,2326,0.399432,116022.0,115034.0,48888.0,45710.0,3178.0,33861.0,13817.0,1210.0
1,chula vista,1323,717,1428,0,224,201,663,0,0,...,4988,0.555245,281850.0,280211.0,91485.0,88373.0,3112.0,56792.0,30800.0,3893.0
2,coronado,27,24,36,0,0,0,54,0,0,...,708,0.223684,22687.0,17283.0,9646.0,7438.0,2208.0,5463.0,4180.0,3.0
3,del mar,9,14,17,8,11,10,43,0,0,...,101,0.680982,3937.0,3937.0,2641.0,1952.0,689.0,1903.0,738.0,0.0
4,el cajon,99,210,235,40,83,123,164,0,4,...,2495,0.239329,105449.0,102949.0,37011.0,35686.0,1325.0,17276.0,17852.0,1883.0


In [4]:
# Inspect columns before validation
print(rhna.columns.tolist())


['jur_clean', 'ent_units_total', 'bp_units_total', 'co_units_total', 'ent_affordable_total', 'bp_affordable_total', 'co_affordable_total', 'project_rows', 'ent_very_low_total', 'ent_low_total', 'ent_moderate_total', 'ent_above_moderate_total', 'bp_very_low_total', 'bp_low_total', 'bp_moderate_total', 'bp_above_moderate_total', 'co_very_low_total', 'co_low_total', 'co_moderate_total', 'co_above_moderate_total', 'ent_affordable_share', 'bp_affordable_share', 'co_affordable_share', 'application_units_total', 'application_classified_total', 'application_unclassified_total', 'application_affordable_total', 'application_rows', 'application_very_low_total', 'application_low_total', 'application_moderate_total', 'application_above_moderate_total', 'application_affordable_share', 'rhna_target_very_low', 'rhna_reported_very_low', 'rhna_remaining_very_low', 'rhna_pct_achieved_very_low', 'rhna_target_low', 'rhna_reported_low', 'rhna_remaining_low', 'rhna_pct_achieved_low', 'rhna_target_moderate', 

In [5]:
# Filter San Diego using actual jurisdiction field
jur_col = "jur_clean" if "jur_clean" in rhna.columns else "jurisdiction"

sd = rhna[
    rhna[jur_col].astype(str).str.contains("san diego", case=False, na=False)
].copy()

sd


,jur_clean,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,ent_low_total,...,rhna_remaining_total,rhna_pct_achieved,population_total,population_household,housing_units_total,occupied_units,vacant_units,single_family_units,multifamily_units,mobile_home_units
13,san diego,3,7848,415,0,2050,21,1838,0,0,...,71503,0.338156,1409429.0,1340735.0,581050.0,541963.0,39087.0,297453.0,277185.0,6412.0
17,unincorporated san diego county,462,932,943,46,335,351,1634,0,23,...,1372,0.915522,511798.0,487275.0,183336.0,172330.0,11006.0,137602.0,30859.0,14875.0


In [6]:
# RHNA metrics available
cols = [
    c for c in [
        "jur_clean","jurisdiction","year",
        "rhna_target_total",
        "rhna_target_very_low",
        "rhna_target_low",
        "rhna_target_moderate",
        "rhna_target_above_moderate",
        "rhna_progress_total",
        "rhna_percent_complete"
    ] if c in sd.columns
]

sd[cols]


,jur_clean,rhna_target_total,rhna_target_very_low,rhna_target_low,rhna_target_moderate,rhna_target_above_moderate
13,san diego,108036,27549,17331,19319,43837
17,unincorporated san diego county,6700,1834,992,1165,2709


In [7]:
# Production stages
stage_cols = [
    c for c in [
        "application_units_total",
        "entitlement_units_total",
        "bp_units_total",
        "co_units_total"
    ] if c in rhna.columns
]

sd[[c for c in [jur_col,"year"] if c in sd.columns] + stage_cols]


,jur_clean,application_units_total,bp_units_total,co_units_total
13,san diego,5560,7848,415
17,unincorporated san diego county,1466,932,943


In [8]:
# Production by housing type
housing_type_file = list(PROCESSED.glob("powerbi_production_by_housing_type*.csv"))[0]
housing_type = pd.read_csv(housing_type_file)

housing_type.head()


,jurisdiction,year,development_stage,housing_type,value,reporting_status,source,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,Carlsbad,2018,Building Permit,2-4 unit building (combined),11.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
1,Carlsbad,2018,Building Permit,5+ unit building,49.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
2,Carlsbad,2018,Building Permit,Accessory dwelling unit,33.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
3,Carlsbad,2018,Building Permit,Mobile home / manufactured home,0.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
4,Carlsbad,2018,Building Permit,Other / source-reported,0.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."


In [9]:
# Housing stock benchmark
benchmark_file = list(PROCESSED.glob("powerbi_housing_stock_benchmark*.csv"))[0]
benchmarks = pd.read_csv(benchmark_file)

benchmarks.head()


,jurisdiction,jur_clean,benchmark_year,benchmark_label,housing_units_total,source,source_dataset,source_variable,source_url,geographic_level,estimate_type,derivation,download_date,limitations
0,Carlsbad,carlsbad,2000,2000 Census,33798.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
1,Chula Vista,chula vista,2000,2000 Census,59495.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
2,Coronado,coronado,2000,2000 Census,9494.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
3,Del Mar,del mar,2000,2000 Census,2557.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
4,El Cajon,el cajon,2000,2000 Census,35190.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...


In [10]:
# DOF annual housing stock
dof_file = list(PROCESSED.glob("powerbi_dof_annual_housing_stock*.csv"))[0]
dof = pd.read_csv(dof_file)

dof.head()


,jurisdiction,jur_clean,year,geographic_level,value,metric,source,source_release,source_year,download_date,unit_of_measurement,limitations
0,Carlsbad,carlsbad,2020,Jurisdiction,47734.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
1,Chula Vista,chula vista,2020,Jurisdiction,87284.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
2,Coronado,coronado,2020,Jurisdiction,9573.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
3,Del Mar,del mar,2020,Jurisdiction,2574.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
4,El Cajon,el cajon,2020,Jurisdiction,36749.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...


In [11]:
# Export validation snapshot
sd.to_csv("workstream1_validation_snapshot.csv", index=False)
print("Saved")


Saved


In [ ]:
# Testing (Ignore)

results = []


# -----------------------------
# RHNA + Production stages
# -----------------------------

sd_row = sd.iloc[0]


rhna_metrics = [
    ("RHNA Allocation - Total", "rhna_target_total"),
    ("RHNA Allocation - Very Low Income", "rhna_target_very_low"),
    ("RHNA Allocation - Low Income", "rhna_target_low"),
    ("RHNA Allocation - Moderate Income", "rhna_target_moderate"),
    ("RHNA Allocation - Above Moderate Income", "rhna_target_above_moderate"),

    ("Applications Submitted", "application_units_total"),
    ("Entitlements", "ent_units_total"),
    ("Building Permits", "bp_units_total"),
    ("Completed Units", "co_units_total"),

    ("RHNA Remaining Units", "rhna_remaining_total"),
    ("RHNA % Complete", "rhna_pct_achieved"),
]


for name, col in rhna_metrics:
    results.append([
        name,
        sd_row[col] if col in sd_row else "NOT AVAILABLE"
    ])



# -----------------------------
# Housing Production by Type
# -----------------------------

ht = housing_type[
    housing_type["jurisdiction"]
    .str.contains("San Diego", case=False, na=False)
].copy()


# choose year
validation_year = 2025

ht = ht[ht["year"] == validation_year]


for stage in ["Building Permit", "Certificate of Occupancy"]:

    temp = ht[
        ht["development_stage"] == stage
    ]

    for _, row in temp.iterrows():

        results.append([
            f"{stage} - {row['housing_type']}",
            row["value"]
        ])




# -----------------------------
# Housing Stock Benchmarks
# -----------------------------

hb = benchmarks[
    benchmarks["jurisdiction"]
    .str.contains("San Diego", case=False, na=False)
]


for _, row in hb.iterrows():

    results.append([
        f"Housing Stock Benchmark - {row['benchmark_year']}",
        row["housing_units_total"]
    ])



# -----------------------------
# DOF Annual Housing Stock
# -----------------------------

dof_sd = dof[
    dof["jurisdiction"]
    .str.contains("San Diego", case=False, na=False)
]


for _, row in dof_sd.iterrows():

    results.append([
        f"DOF Housing Stock {row['year']}",
        row["value"]
    ])



# FINAL TABLE

validation_table = pd.DataFrame(
    results,
    columns=[
        "Metric",
        "Notebook Value"
    ]
)

validation_table

,Metric,Notebook Value
0,RHNA Allocation - Total,108036.0
1,RHNA Allocation - Very Low Income,27549.0
2,RHNA Allocation - Low Income,17331.0
3,RHNA Allocation - Moderate Income,19319.0
4,RHNA Allocation - Above Moderate Income,43837.0
...,...,...
222,DOF Housing Stock 2024,14791.0
223,DOF Housing Stock 2024,44717.0
224,DOF Housing Stock 2025,6412.0
225,DOF Housing Stock 2025,14875.0


In [ ]:
# Testing (Ignore)

results = []


# -----------------------------
# RHNA + Production stages
# -----------------------------

sd_row = sd.iloc[0]


rhna_metrics = [
    ("RHNA Allocation - Total", "rhna_target_total"),
    ("RHNA Allocation - Very Low Income", "rhna_target_very_low"),
    ("RHNA Allocation - Low Income", "rhna_target_low"),
    ("RHNA Allocation - Moderate Income", "rhna_target_moderate"),
    ("RHNA Allocation - Above Moderate Income", "rhna_target_above_moderate"),

    ("Applications Submitted", "application_units_total"),
    ("Entitlements", "ent_units_total"),
    ("Building Permits", "bp_units_total"),
    ("Completed Units", "co_units_total"),

    ("RHNA Remaining Units", "rhna_remaining_total"),
    ("RHNA % Complete", "rhna_pct_achieved"),
]


for name, col in rhna_metrics:
    results.append([
        name,
        sd_row[col] if col in sd_row else "NOT AVAILABLE"
    ])



# -----------------------------
# Housing Production by Type
# -----------------------------

ht = housing_type[
    housing_type["jurisdiction"]
    .str.contains("San Diego", case=False, na=False)
].copy()


# choose year
validation_year = 2025

ht = ht[ht["year"] == validation_year]


for stage in ["Building Permit", "Certificate of Occupancy"]:

    temp = ht[
        ht["development_stage"] == stage
    ]

    for _, row in temp.iterrows():

        results.append([
            f"{stage} - {row['housing_type']}",
            row["value"]
        ])




# -----------------------------
# Housing Stock Benchmarks
# -----------------------------

benchmark_years = [2000, 2010, 2021, 2024]

for year in benchmark_years:

    row = benchmarks[
        (benchmarks["benchmark_year"] == year) &
        (benchmarks["jurisdiction"]
         .str.contains("San Diego", case=False, na=False))
    ]

    results.append([
        f"Housing Stock Benchmark - {year}",
        row["housing_units_total"].iloc[0]
    ])



# -----------------------------
# DOF Annual Housing Stock
# -----------------------------

for year in [2018,2019,2020,2021,2022,2023,2024,2025]:

    row = dof[
        (dof["year"] == year) &
        (dof["jurisdiction"]
         .str.contains("San Diego", case=False, na=False))
    ]

    results.append([
        f"DOF Housing Stock - {year}",
        row["value"].sum()
    ])



# FINAL TABLE

validation_table = pd.DataFrame(
    results,
    columns=[
        "Metric",
        "Notebook Value"
    ]
)

# Format values for validation output

validation_table["Notebook Value"] = (
    validation_table["Notebook Value"]
    .apply(lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else x)
)

validation_table

,Metric,Notebook Value
0,RHNA Allocation - Total,"108,036"
1,RHNA Allocation - Very Low Income,"27,549"
2,RHNA Allocation - Low Income,"17,331"
3,RHNA Allocation - Moderate Income,"19,319"
4,RHNA Allocation - Above Moderate Income,"43,837"
5,Applications Submitted,"5,560"
6,Entitlements,3
7,Building Permits,"7,848"
8,Completed Units,415
9,RHNA Remaining Units,"71,503"


In [24]:
# Show the exact dataframe names + columns that exist

for name in ["rhna", "production", "benchmarks", "dof"]:
    if name in globals():
        print("\n======================")
        print(name)
        print("======================")
        print("Shape:", globals()[name].shape)
        print("Columns:")
        print(list(globals()[name].columns))


rhna
Shape: (19, 61)
Columns:
['jur_clean', 'ent_units_total', 'bp_units_total', 'co_units_total', 'ent_affordable_total', 'bp_affordable_total', 'co_affordable_total', 'project_rows', 'ent_very_low_total', 'ent_low_total', 'ent_moderate_total', 'ent_above_moderate_total', 'bp_very_low_total', 'bp_low_total', 'bp_moderate_total', 'bp_above_moderate_total', 'co_very_low_total', 'co_low_total', 'co_moderate_total', 'co_above_moderate_total', 'ent_affordable_share', 'bp_affordable_share', 'co_affordable_share', 'application_units_total', 'application_classified_total', 'application_unclassified_total', 'application_affordable_total', 'application_rows', 'application_very_low_total', 'application_low_total', 'application_moderate_total', 'application_above_moderate_total', 'application_affordable_share', 'rhna_target_very_low', 'rhna_reported_very_low', 'rhna_remaining_very_low', 'rhna_pct_achieved_very_low', 'rhna_target_low', 'rhna_reported_low', 'rhna_remaining_low', 'rhna_pct_achieved

In [25]:
# ============================================================
# FINAL WORKSTREAM 1 VALIDATION OUTPUT
# Uses existing validation notebook dataframes
# ============================================================

results = []

TARGET_JUR = "San Diego"


# ------------------------------------------------------------
# 1. RHNA + APR metrics
# ------------------------------------------------------------

sd_rhna = rhna[
    rhna["jur_clean"].str.contains(
        TARGET_JUR,
        case=False,
        na=False
    )
].iloc[0]


rhna_metrics = [

    ("RHNA Allocation - Total",
     "rhna_target_total"),

    ("RHNA Allocation - Very Low Income",
     "rhna_target_very_low"),

    ("RHNA Allocation - Low Income",
     "rhna_target_low"),

    ("RHNA Allocation - Moderate Income",
     "rhna_target_moderate"),

    ("RHNA Allocation - Above Moderate Income",
     "rhna_target_above_moderate"),


    ("RHNA Qualifying Units - Total",
     "rhna_units_reported_total"),


    ("RHNA Remaining Units - Total",
     "rhna_remaining_total"),


    ("RHNA % Complete",
     "rhna_pct_achieved"),


    ("Applications Submitted",
     "application_units_total"),

    ("Entitlements",
     "ent_units_total"),

    ("Building Permits",
     "bp_units_total"),

    ("Completed Units",
     "co_units_total"),

]


for metric, column in rhna_metrics:

    results.append(
        [
            metric,
            sd_rhna[column]
        ]
    )



# ------------------------------------------------------------
# 2. Housing Stock from RHNA dataframe
# ------------------------------------------------------------

stock_metrics = [

    ("Population",
     "population_total"),

    ("Housing Units Total",
     "housing_units_total"),

    ("Occupied Housing Units",
     "occupied_units"),

    ("Vacant Housing Units",
     "vacant_units"),

    ("Single-Family Housing Units",
     "single_family_units"),

    ("Multifamily Housing Units",
     "multifamily_units"),

    ("Mobile Home Units",
     "mobile_home_units"),

]


for metric, column in stock_metrics:

    results.append(
        [
            metric,
            sd_rhna[column]
        ]
    )



# ------------------------------------------------------------
# 3. Housing Stock Benchmarks
# ------------------------------------------------------------

benchmark_sd = benchmarks[
    benchmarks["jur_clean"].str.contains(
        TARGET_JUR,
        case=False,
        na=False
    )
]


for year in [2000, 2010, 2021, 2024]:

    row = benchmark_sd[
        benchmark_sd["benchmark_year"] == year
    ]

    if len(row) > 0:

        results.append(
            [
                f"Housing Stock Benchmark - {year}",
                row["housing_units_total"].iloc[0]
            ]
        )



# ------------------------------------------------------------
# 4. DOF Annual Housing Stock
# ------------------------------------------------------------

dof_sd = dof[
    dof["jur_clean"].str.contains(
        TARGET_JUR,
        case=False,
        na=False
    )
]


# Only total housing units
dof_total = dof_sd[
    dof_sd["metric"]
    .str.contains(
        "total",
        case=False,
        na=False
    )
]


for _, row in dof_total.iterrows():

    results.append(
        [
            f"DOF Housing Stock - {row['year']}",
            row["value"]
        ]
    )



# ------------------------------------------------------------
# Final dataframe
# ------------------------------------------------------------

validation_table = pd.DataFrame(
    results,
    columns=[
        "Metric",
        "Notebook Value"
    ]
)


validation_table

,Metric,Notebook Value
0,RHNA Allocation - Total,108036.0
1,RHNA Allocation - Very Low Income,27549.0
2,RHNA Allocation - Low Income,17331.0
3,RHNA Allocation - Moderate Income,19319.0
4,RHNA Allocation - Above Moderate Income,43837.0
...,...,...
72,DOF Housing Stock - 2024,30828.0
73,DOF Housing Stock - 2024,474715.0
74,DOF Housing Stock - 2025,277185.0
75,DOF Housing Stock - 2025,30859.0


In [27]:
# Show all rows + normal number formatting

pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

validation_table

,Metric,Notebook Value
0,RHNA Allocation - Total,"108,036"
1,RHNA Allocation - Very Low Income,"27,549"
2,RHNA Allocation - Low Income,"17,331"
3,RHNA Allocation - Moderate Income,"19,319"
4,RHNA Allocation - Above Moderate Income,"43,837"
5,RHNA Qualifying Units - Total,"36,533"
6,RHNA Remaining Units - Total,"71,503"
7,RHNA % Complete,0
8,Applications Submitted,"5,560"
9,Entitlements,3
